# V2_01 — Stage 1: Full-Data Training

Train XGBoost, CatBoost, and LightGBM on **all 126.8M rows** — no sampling, no regional batching.

**Runtime:** T4 GPU + High-RAM | ~4-6 hrs | ~8-12 CU

**Prerequisite:** Gold parquets uploaded to `My Drive/AllowanceMap/V2/gold/`

In [ ]:
# ── Cell 1: Environment Setup ──────────────────────────────────────────────
import os, subprocess, sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'mlflow>=2.12', 'catboost>=1.2', 'lightgbm>=4.3', 'databricks-sdk>=0.20'])

from google.colab import drive, userdata
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/AllowanceMap/V2'
GOLD_DIR   = f'{DRIVE_ROOT}/gold'
ARTIFACTS  = f'{DRIVE_ROOT}/v2_artifacts'
os.makedirs(f'{ARTIFACTS}/models', exist_ok=True)
os.makedirs(f'{ARTIFACTS}/predictions', exist_ok=True)
os.makedirs(f'{ARTIFACTS}/plots', exist_ok=True)

os.environ['DATABRICKS_HOST']  = 'https://dbc-d709cbb6-fe84.cloud.databricks.com'
os.environ['DATABRICKS_TOKEN'] = userdata.get('DATABRICKS_TOKEN')

import mlflow, requests
mlflow.set_tracking_uri('databricks')
resp = requests.get(
    f"{os.environ['DATABRICKS_HOST']}/api/2.0/preview/scim/v2/Me",
    headers={'Authorization': f"Bearer {os.environ['DATABRICKS_TOKEN']}"},
    timeout=10,
)
resp.raise_for_status()
USER_HOME = f"/Users/{resp.json()['userName']}"
mlflow.set_experiment(f'{USER_HOME}/medicare_models')
print(f'MLflow: {USER_HOME}/medicare_models')

In [ ]:
# ── Cell 2: Constants & Data Loading ───────────────────────────────────────
import glob
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
import time

TARGET = 'Avg_Mdcr_Alowd_Amt'

FEATURES = [
    'Rndrng_Prvdr_Type_idx', 'Rndrng_Prvdr_State_Abrvtn_idx',
    'HCPCS_Cd_idx', 'hcpcs_bucket', 'place_of_srvc_flag',
    'Bene_Avg_Risk_Scre', 'log_srvcs', 'log_benes',
    'Avg_Sbmtd_Chrg', 'srvcs_per_bene',
    'specialty_bucket', 'pos_bucket', 'hcpcs_target_enc',
]

CAT_FEATURE_NAMES = [
    'Rndrng_Prvdr_Type_idx', 'Rndrng_Prvdr_State_Abrvtn_idx',
    'HCPCS_Cd_idx', 'hcpcs_bucket', 'place_of_srvc_flag',
    'specialty_bucket', 'pos_bucket',
]

def get_cat_indices(features):
    return [i for i, f in enumerate(features) if f in CAT_FEATURE_NAMES]

def make_pool(X_df, y, features, cat_indices):
    """Create CatBoost Pool from DataFrame with proper categorical handling."""
    X = X_df[features].copy()
    for idx in cat_indices:
        col = features[idx]
        X[col] = X[col].astype(int)
    return Pool(X, label=y, cat_features=cat_indices, feature_names=features)

def log_metrics(y_true_log, y_pred_log, prefix=''):
    """Compute and log dollar-scale metrics from log1p predictions."""
    y_true = np.expm1(y_true_log)
    y_pred = np.expm1(y_pred_log)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mlflow.log_metrics({f'{prefix}mae': mae, f'{prefix}rmse': rmse, f'{prefix}r2': r2})
    print(f'  {prefix}MAE=${mae:.2f}  RMSE=${rmse:.2f}  R²={r2:.4f}')
    return mae, rmse, r2

In [ ]:
# ── Cell 3: Load Full Dataset ──────────────────────────────────────────────
print('Loading all gold parquets...')
t0 = time.time()

load_cols = list(dict.fromkeys(FEATURES + [TARGET, 'year']))
parts = []
for f in sorted(glob.glob(os.path.join(GOLD_DIR, '*.parquet'))):
    avail = set(pq.read_schema(f).names)
    cols = [c for c in load_cols if c in avail]
    df = pd.read_parquet(f, columns=cols).dropna(subset=FEATURES + [TARGET])
    parts.append(df)
    
df_all = pd.concat(parts, ignore_index=True)
del parts  # free memory

print(f'Loaded {len(df_all):,} rows in {time.time()-t0:.0f}s')
print(f'Memory: {df_all.memory_usage(deep=True).sum() / 1e9:.1f} GB')
print(f'Columns: {list(df_all.columns)}')
print(f'Year range: {df_all["year"].min()} - {df_all["year"].max()}')

In [ ]:
# ── Cell 4: Prepare Train/Test Split ───────────────────────────────────────
# Keep as DataFrame for CatBoost Pool compatibility, extract numpy for XGB/LGB
import gc

y_all = np.log1p(df_all[TARGET].astype('float64').values)
X_df = df_all[FEATURES].copy()
del df_all; gc.collect()

# Split indices first, then slice both DataFrame and arrays
from sklearn.model_selection import train_test_split
idx_train, idx_test = train_test_split(
    np.arange(len(y_all)), test_size=0.2, random_state=42
)

# Numpy arrays for XGB and LGB
X_all_np = X_df.astype('float64').values
X_train = X_all_np[idx_train]
X_test  = X_all_np[idx_test]
y_train = y_all[idx_train]
y_test  = y_all[idx_test]
del X_all_np, y_all; gc.collect()

# DataFrame slices for CatBoost (keeps proper dtypes)
X_df_train = X_df.iloc[idx_train].reset_index(drop=True)
X_df_test  = X_df.iloc[idx_test].reset_index(drop=True)
del X_df; gc.collect()

print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')

## XGBoost — Full Data, 1000 Rounds

In [ ]:
# ── Cell 5: XGBoost ────────────────────────────────────────────────────────
import xgboost as xgb

XGB_PARAMS = {
    'objective':        'reg:squarederror',
    'learning_rate':    0.05,
    'max_depth':        6,
    'subsample':        0.8,
    'colsample_bytree': 0.8,
    'tree_method':      'hist',
    'device':           'cuda',
    'seed':             42,
    'max_bin':          256,
}

print('Building DMatrix...')
t0 = time.time()
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=FEATURES)
dtest  = xgb.DMatrix(X_test,  label=y_test,  feature_names=FEATURES)
print(f'DMatrix built in {time.time()-t0:.0f}s')

print('Training XGBoost...')
t0 = time.time()
xgb_booster = xgb.train(
    XGB_PARAMS, dtrain, num_boost_round=1000,
    evals=[(dtest, 'val')], early_stopping_rounds=50, verbose_eval=100,
)
xgb_time = time.time() - t0
print(f'XGBoost done in {xgb_time:.0f}s ({xgb_booster.best_iteration} rounds)')

# Log to MLflow
with mlflow.start_run(run_name='xgb_v2_full_colab'):
    mlflow.log_params({
        **XGB_PARAMS,
        'n_rounds': 1000,
        'early_stopping': 50,
        'best_iteration': xgb_booster.best_iteration,
        'source': 'colab',
        'strategy': 'full_dmatrix',
        'target_transform': 'log1p',
        'sample_frac': 1.0,
        'split_strategy': 'random',
        'n_features': len(FEATURES),
        'ablation_avg_submitted_charge': False,
        'version': 'v2',
        'training_time_sec': int(xgb_time),
        'n_train': len(X_train),
        'n_test': len(X_test),
    })
    y_pred_xgb = xgb_booster.predict(dtest)
    log_metrics(y_test, y_pred_xgb, prefix='test_')
    importances = xgb_booster.get_score(importance_type='gain')
    mlflow.log_dict(importances, 'feature_importances.json')
    mlflow.xgboost.log_model(xgb_booster, artifact_path='xgb_model')

# Save to Drive
xgb_booster.save_model(f'{ARTIFACTS}/models/xgb_v2_full.json')
print(f'Saved to {ARTIFACTS}/models/xgb_v2_full.json')

# Free DMatrix memory
del dtrain; gc.collect()

## CatBoost — Full Data, 1000 Iterations

In [ ]:
# ── Cell 6: CatBoost ──────────────────────────────────────────────────────
from catboost import CatBoostRegressor, Pool

CB_PARAMS = {
    'loss_function':     'RMSE',
    'learning_rate':     0.05,
    'depth':             6,
    'subsample':         0.8,
    'rsm':               0.8,
    'task_type':         'GPU',
    'random_seed':       42,
    'verbose':           100,
    'allow_writing_files': False,
    'iterations':        1000,
}

cat_idx = get_cat_indices(FEATURES)
print(f'Categorical feature indices: {cat_idx}')
print(f'Categorical features: {[FEATURES[i] for i in cat_idx]}')

# Use DataFrame-based Pool (CatBoost rejects float64 numpy with cat_features)
print('Building CatBoost Pools...')
t0 = time.time()
pool_train = make_pool(X_df_train, y_train, FEATURES, cat_idx)
pool_test  = make_pool(X_df_test,  y_test,  FEATURES, cat_idx)
print(f'Pools built in {time.time()-t0:.0f}s')

print('Training CatBoost...')
t0 = time.time()
cb_model = CatBoostRegressor(**CB_PARAMS)
cb_model.fit(pool_train, eval_set=pool_test, early_stopping_rounds=50, use_best_model=True)
cb_time = time.time() - t0
print(f'CatBoost done in {cb_time:.0f}s ({cb_model.best_iteration_} iterations)')

# Log to MLflow
with mlflow.start_run(run_name='catboost_v2_full_colab'):
    mlflow.log_params({
        **{k: v for k, v in CB_PARAMS.items() if k != 'verbose'},
        'best_iteration': cb_model.best_iteration_,
        'source': 'colab',
        'strategy': 'full_pool',
        'target_transform': 'log1p',
        'sample_frac': 1.0,
        'split_strategy': 'random',
        'n_features': len(FEATURES),
        'n_cat_features': len(cat_idx),
        'ablation_avg_submitted_charge': False,
        'version': 'v2',
        'training_time_sec': int(cb_time),
        'n_train': len(X_train),
        'n_test': len(X_test),
    })
    y_pred_cb = cb_model.predict(pool_test)
    log_metrics(y_test, y_pred_cb, prefix='test_')
    importances = dict(zip(FEATURES, cb_model.get_feature_importance()))
    mlflow.log_dict(importances, 'feature_importances.json')
    mlflow.catboost.log_model(cb_model, artifact_path='catboost_model')

# Save to Drive
cb_model.save_model(f'{ARTIFACTS}/models/catboost_v2_full.cbm')
print(f'Saved to {ARTIFACTS}/models/catboost_v2_full.cbm')

# Free Pool memory
del pool_train; gc.collect()

## LightGBM — Full Data, 1000 Rounds, GOSS

In [ ]:
# ── Cell 7: LightGBM ──────────────────────────────────────────────────────
import lightgbm as lgb

LGB_PARAMS = {
    'objective':            'regression',
    'metric':               'rmse',
    'learning_rate':        0.05,
    'num_leaves':           63,
    'max_depth':            -1,
    'subsample':            0.8,
    'colsample_bytree':     0.8,
    'min_child_samples':    20,
    'device':               'gpu',
    'seed':                 42,
    'verbose':              -1,
    'boosting_type':        'gbdt',
    'data_sample_strategy': 'goss',
}

cat_cols = [FEATURES[i] for i in get_cat_indices(FEATURES)]
print(f'LightGBM categorical features: {cat_cols}')

print('Building LightGBM Datasets...')
t0 = time.time()
ds_train = lgb.Dataset(X_train, label=y_train, feature_name=FEATURES,
                       categorical_feature=cat_cols, free_raw_data=True)
ds_test  = lgb.Dataset(X_test, label=y_test, feature_name=FEATURES,
                       categorical_feature=cat_cols, reference=ds_train, free_raw_data=True)
print(f'Datasets built in {time.time()-t0:.0f}s')

print('Training LightGBM...')
t0 = time.time()
lgb_booster = lgb.train(
    LGB_PARAMS, ds_train, num_boost_round=1000,
    valid_sets=[ds_test], callbacks=[
        lgb.early_stopping(50),
        lgb.log_evaluation(100),
    ],
)
lgb_time = time.time() - t0
print(f'LightGBM done in {lgb_time:.0f}s ({lgb_booster.best_iteration} rounds)')

# Log to MLflow
with mlflow.start_run(run_name='lgbm_v2_full_colab'):
    mlflow.log_params({
        **{k: v for k, v in LGB_PARAMS.items() if k != 'verbose'},
        'n_rounds': 1000,
        'early_stopping': 50,
        'best_iteration': lgb_booster.best_iteration,
        'source': 'colab',
        'strategy': 'full_dataset',
        'target_transform': 'log1p',
        'sample_frac': 1.0,
        'split_strategy': 'random',
        'n_features': len(FEATURES),
        'n_cat_features': len(cat_cols),
        'ablation_avg_submitted_charge': False,
        'version': 'v2',
        'training_time_sec': int(lgb_time),
        'n_train': len(X_train),
        'n_test': len(X_test),
    })
    y_pred_lgb = lgb_booster.predict(X_test)
    log_metrics(y_test, y_pred_lgb, prefix='test_')
    importances = dict(zip(FEATURES, lgb_booster.feature_importance(importance_type='gain')))
    mlflow.log_dict(importances, 'feature_importances.json')
    mlflow.lightgbm.log_model(lgb_booster, artifact_path='lgbm_model')

# Save to Drive
lgb_booster.save_model(f'{ARTIFACTS}/models/lgbm_v2_full.txt')
print(f'Saved to {ARTIFACTS}/models/lgbm_v2_full.txt')

del ds_train, ds_test; gc.collect()

## Summary

In [ ]:
# ── Cell 8: Summary Table ──────────────────────────────────────────────────
print('\n' + '='*70)
print('V2_01 STAGE 1 FULL-DATA TRAINING COMPLETE')
print('='*70)
print(f'Dataset: {len(X_train) + len(X_test):,} total rows (no sampling)')
print(f'Features: {len(FEATURES)} (with Avg_Sbmtd_Chrg)')
print(f'Split: 80/20 random, seed=42')
print()

# Recompute metrics for summary table
for name, y_pred in [('XGBoost', y_pred_xgb), ('CatBoost', y_pred_cb), ('LightGBM', y_pred_lgb)]:
    yt = np.expm1(y_test)
    yp = np.expm1(y_pred)
    mae  = mean_absolute_error(yt, yp)
    rmse = np.sqrt(mean_squared_error(yt, yp))
    r2   = r2_score(yt, yp)
    print(f'{name:12s}  MAE=${mae:8.2f}  RMSE=${rmse:8.2f}  R²={r2:.4f}')

print()
print('V1 baselines for reference:')
print('  RF (V1)       MAE=$12.04   RMSE=$22.73   R²=0.8843  (0.3 sample, batch)')
print('  XGB (V1)      MAE=$11.83   RMSE=$21.18   R²=0.8331  (0.3 sample, batch)')
print()
print('All models logged to Databricks MLflow.')
print(f'Artifacts saved to {ARTIFACTS}/models/')